In [8]:
!pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 2.6 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 2.0 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.1 MB/s  0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

def scrapping2pdf(link, titulo, descripcion):
    nombre_pdf = link.rstrip("/").split("/")[-1]

    # Configuración de PDF
    doc = SimpleDocTemplate(
        f"../doc_pdf/{nombre_pdf.title()}_Basecamp.pdf",
        pagesize=A4,
        rightMargin=40, leftMargin=40,
        topMargin=40, bottomMargin=40
    )
    styles = getSampleStyleSheet()
    story = []

    # Portada
    story.append(Paragraph(f"<para align='center'><b>{titulo}</b></para>", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"<para align='center'>{descripcion}</para>", styles["Normal"]))
    story.append(Paragraph(f"<para align='center'>Libro recopilado de {link}</para>", styles["Normal"]))
    story.append(PageBreak())

    # Obtener lista de capítulos
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    nombre_pdf = link.rstrip("/").split("/")[-1]
    chapter_links = []
    for a in soup.select("a[href]"):
        href = a["href"]
        if href.startswith(f"/{nombre_pdf}/") and href != f"/{nombre_pdf}/":
            chapter_links.append("https://basecamp.com" + href)

    chapter_links = sorted(set(chapter_links))

    # Índice
    story.append(Paragraph("<b>Index</b>", styles["Heading2"]))
    for i, ch_link in enumerate(chapter_links, 1):
        res = requests.get(ch_link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")
        title = chapter_soup.find("h1").get_text(strip=True)
        story.append(Paragraph(f"Chapter {i}. {title}", styles["Normal"]))
    story.append(PageBreak())

    # Capítulos
    for i, ch_link in enumerate(chapter_links, 1):
        res = requests.get(ch_link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")

        # Título del capítulo
        title = chapter_soup.find("h1").get_text(strip=True)

        content_elem = chapter_soup.select_one("div.content")
        if not content_elem:
            continue

        # Limpiar footers/publicidad
        for unwanted in content_elem.find_all(["footer"], recursive=True):
            unwanted.decompose()
        for p in content_elem.find_all("p"):
            if "We made" in p.get_text() or "Copyright" in p.get_text():
                p.decompose()
        for a in content_elem.find_all("a"):
            if not a.get_text(strip=True):
                a.string = a["href"]

        # Agregar capítulo
        story.append(Paragraph(f"Chapter {i}: {title}", styles["Heading1"]))
        story.append(Spacer(1, 12))

        for p in content_elem.find_all("p"):
            text = p.get_text(strip=True)
            if text:
                story.append(Paragraph(text, styles["Normal"]))
                story.append(Spacer(1, 6))

        story.append(PageBreak())

    # Generar PDF
    doc.build(story)
    print(f"✅ Libro {nombre_pdf.title()}_Basecamp.pdf generado correctamente en ../doc_pdf/")


In [3]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/gettingreal"
titulo= "Getting Real - Basecamp "
descripcion ="The smarter, faster, easier way to build a successful web application"

scrapping2pdf(link, titulo, descripcion)

✅ Libro Gettingreal_Basecamp.pdf generado correctamente en ../doc_pdf/


In [24]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/shapeup"
titulo= "Shape Up - Basecamp "
descripcion ="Stop Running in Circles and Ship Work that Matters"

scrapping2pdf(link, titulo, descripcion)

✅ Libro shapeup.pdf generado correctamente en ../doc_pdf/


Prueba para SHAPE UP

In [6]:
import requests
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

def scrapping2pdf(link, titulo, descripcion):
    nombre_pdf = link.rstrip("/").split("/")[-1]

    # Configuración de PDF
    doc = SimpleDocTemplate(
        f"../doc_pdf/{nombre_pdf}_v2.pdf",
        pagesize=A4,
        rightMargin=40, leftMargin=40,
        topMargin=40, bottomMargin=40
    )
    styles = getSampleStyleSheet()
    story = []

    # Portada
    story.append(Paragraph(f"<para align='center'><b>{titulo}</b></para>", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"<para align='center'>{descripcion}</para>", styles["Normal"]))
    story.append(Paragraph(f"<para align='center'>Libro recopilado de {link}</para>", styles["Normal"]))
    story.append(PageBreak())

    # Obtener contenido
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    # analizamos la estructura
    print(soup.prettify()[:10000])  # imprime solo los primeros 2000 caracteres del HTML

    # # Selecciona el contenido principal (Shape Up está en <div class="book">)
    # book = soup.select_one("main.book")

    # # Extraemos títulos y párrafos en orden
    # elements = book.find_all(["h1", "h2", "h3", "p", "ul", "ol"])

    # # Índice (basado solo en h1 y h2)
    # story.append(Paragraph("<b>Index</b>", styles["Heading2"]))
    # toc = []
    # for el in elements:
    #     if el.name in ["h1", "h2"]:
    #         title = el.get_text(strip=True)
    #         toc.append(title)
    #         story.append(Paragraph(title, styles["Normal"]))
    # story.append(PageBreak())

    # # Contenido
    # for el in elements:
    #     if el.name == "h1":
    #         story.append(Paragraph(el.get_text(strip=True), styles["Heading1"]))
    #         story.append(Spacer(1, 12))
    #     elif el.name == "h2":
    #         story.append(Paragraph(el.get_text(strip=True), styles["Heading2"]))
    #         story.append(Spacer(1, 12))
    #     elif el.name == "h3":
    #         story.append(Paragraph(el.get_text(strip=True), styles["Heading3"]))
    #         story.append(Spacer(1, 12))
    #     elif el.name == "p":
    #         text = el.get_text(strip=True)
    #         if text:
    #             story.append(Paragraph(text, styles["Normal"]))
    #             story.append(Spacer(1, 6))
    #     elif el.name in ["ul", "ol"]:
    #         for li in el.find_all("li"):
    #             story.append(Paragraph("• " + li.get_text(strip=True), styles["Normal"]))
    #             story.append(Spacer(1, 4))

    # # Generar PDF
    # doc.build(story)
    # print(f"✅ Libro {nombre_pdf}_v2.pdf generado correctamente en ../doc_pdf/")


In [7]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/shapeup"
titulo= "Shape Up - Basecamp "
descripcion ="Stop Running in Circles and Ship Work that Matters"

scrapping2pdf(link, titulo, descripcion)

<!DOCTYPE html>
<html lang="en">
 <head>
  <!-- online collaboration: Basecamp -->
  <meta charset="utf-8"/>
  <link href="/assets/css/web-book.css" rel="stylesheet" type="text/css"/>
  <link href="https://use.typekit.net/xig7qap.css" rel="stylesheet"/>
  <script src="/assets/js/packages/popper.js">
  </script>
  <script src="/assets/js/web-book.js" type="module">
  </script>
  <link href="/assets/images/general/favicon-32.png" rel="icon" sizes="32x32" type="image/png"/>
  <link href="/assets/images/general/favicon-16.png" rel="icon" sizes="16x16" type="image/png"/>
  <link href="/assets/images/general/apple-touch-icon.png" rel="apple-touch-icon"/>
  <title>
   Shape Up: Stop Running in Circles and Ship Work that Matters
  </title>
  <meta content="Shape Up will help you break free of “best practices” that aren’t really working, think deeper about the right problems, and start shipping meaningful projects your team can celebrate." name="description"/>
  <meta content="#000000" name="th

Identificar la estructura

In [9]:
import requests
from bs4 import BeautifulSoup

url = "https://basecamp.com/shapeup"
response = requests.get(url)
response.encoding = "utf-8"
soup = BeautifulSoup(response.text, "html.parser")

# Ver las primeras cabeceras y párrafos para identificar estructura
for tag in soup.find_all(["h1", "h2", "h3", "p"], limit=40):
    print(tag.name, ":", tag.get_text(strip=True))


p : Heads up!This page uses features your browser doesn’t support. Try a modern browser likeFirefoxorChromefor the best experience.
p : ← Basecamp.com
h1 : Shape Up
p : Stop Running in Circlesand Ship Work that Mattersby Ryan SingerBuy the print editionStart reading →IntroductionForeword by Jason FriedAcknowledgementsChapter 1IntroductionGrowing painsSix-week cyclesShaping the workMaking teams responsibleTargeting riskHow this book is organizedPart 1: ShapingChapter 2Principles of ShapingWireframes are too concreteWords are too abstractCase study: The Dot Grid CalendarProperty 1: It’s roughProperty 2: It’s solvedProperty 3: It’s boundedWho shapesTwo tracksSteps to shapingChapter 3Set BoundariesSetting the appetiteFixed time, variable scope"Good" is relativeResponding to raw ideasNarrow down the problemCase study: Defining "calendar"Watch out for grab-bagsBoundaries in placeChapter 4Find the ElementsMove at the right speedBreadboardingFat marker sketchesElements are the outputRoom for d

In [13]:
import requests
from bs4 import BeautifulSoup

def get_shapeup_toc(base_url: str = "https://basecamp.com/shapeup"):
    """
    Devuelve un dict con la jerarquía del índice, con el formato:
    {
      "Introduction": [
          "Foreword by Jason Fried",
          "Acknowledgements",
          "Chapter 1",
          [
              "Introduction",
              "Growing pains",
              ...
          ],
      ],
      "Part 1: Shaping": [
          "Chapter 2",
          [
              "Principles of Shaping",
              "Wireframes are too concrete",
              ...
          ],
          ...
      ],
      ...
    }
    """
    res = requests.get(base_url)
    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    toc = {}

    # Cada "parte" del índice está dentro de div.toc__part
    for part in soup.select("div.toc > div.toc__part"):
        part_title_el = part.select_one("h2.toc__part-title")
        if not part_title_el:
            # Si por alguna razón no hay título, saltamos
            continue
        part_title = part_title_el.get_text(strip=True)

        items = []
        # Cada capítulo (o entrada) está en li.toc__chapter
        for li in part.select("ul.toc__chapters > li.toc__chapter"):
            ch_num_el = li.select_one("p.toc__chapter-number")  # "Chapter X" (opcional)
            ch_title_a = li.select_one("h3.toc__chapter-title a")  # Título (siempre debería existir)

            # Sub-secciones dentro del capítulo
            section_titles = [
                a.get_text(strip=True)
                for a in li.select("ul.toc__sections li.toc__section a")
            ]

            # Hay dos casos:
            # 1) Entradas sin "Chapter X" (Foreword, Acknowledgements) -> item simple (string)
            # 2) Entradas con "Chapter X" -> agregamos "Chapter X" y luego una lista
            #    con [Título del capítulo, subsecciones...]
            if ch_num_el:
                items.append(ch_num_el.get_text(strip=True))  # "Chapter N"
                sublist = []
                if ch_title_a:
                    sublist.append(ch_title_a.get_text(strip=True))
                if section_titles:
                    sublist.extend(section_titles)
                items.append(sublist)
            else:
                # Foreword / Acknowledgements, etc.
                if ch_title_a:
                    items.append(ch_title_a.get_text(strip=True))

        toc[part_title] = items

    return toc

toc = get_shapeup_toc()
# Ejemplo: imprimir las primeras llaves y algunos elementos
for section, lst in list(toc.items())[:2]:
    print(section)
    for x in lst[:6]:
        print("  ", x)


Introduction
   Foreword by Jason Fried
   Acknowledgements
   Chapter 1
   ['Introduction', 'Growing pains', 'Six-week cycles', 'Shaping the work', 'Making teams responsible', 'Targeting risk', 'How this book is organized']
Part 1: Shaping
   Chapter 2
   ['Principles of Shaping', 'Wireframes are too concrete', 'Words are too abstract', 'Case study: The Dot Grid Calendar', 'Property 1: It’s rough', 'Property 2: It’s solved', 'Property 3: It’s bounded', 'Who shapes', 'Two tracks', 'Steps to shaping']
   Chapter 3
   ['Set Boundaries', 'Setting the appetite', 'Fixed time, variable scope', '"Good" is relative', 'Responding to raw ideas', 'Narrow down the problem', 'Case study: Defining "calendar"', 'Watch out for grab-bags', 'Boundaries in place']
   Chapter 4
   ['Find the Elements', 'Move at the right speed', 'Breadboarding', 'Fat marker sketches', 'Elements are the output', 'Room for designers', 'Not deliverable yet', 'No conveyor belt']


In [43]:
import requests
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.enums import TA_CENTER, TA_LEFT



def scrape_shapeup_to_pdf(toc):
    base_url = "https://basecamp.com/shapeup"
    response = requests.get(base_url)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    # Definir estilos
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("Title", parent=styles["Title"], alignment=TA_CENTER, fontSize=18, spaceAfter=20)
    chapter_style = ParagraphStyle("Chapter", parent=styles["Heading2"], fontSize=14, spaceAfter=10)
    section_style = ParagraphStyle("Section", parent=styles["Normal"], fontSize=12, leftIndent=20, spaceAfter=6)
    content_style = ParagraphStyle("Content", parent=styles["Normal"], fontSize=11, spaceAfter=12, alignment=TA_LEFT)

    # Documento PDF
    doc = SimpleDocTemplate("../doc_pdf/Shapeup_Basecamp.pdf", pagesize=LETTER)
    story = []

    # Título principal
    story.append(Paragraph("Shape Up: Stop Running in Circles and Ship Work that Matters", title_style))
    story.append(Spacer(1, 20))


    # Generar índice en PDF
    story.append(Paragraph("Table of Contents", styles["Heading1"]))
    for section, items in toc.items():
        story.append(Paragraph(section, chapter_style))
        for item in items:
            if isinstance(item, str):
                story.append(Paragraph(f"- {item}", section_style))
            elif isinstance(item, list):
                for sub in item:
                    story.append(Paragraph(f"   • {sub}", section_style))
    story.append(PageBreak())

    # Palabras/frases a excluir
    blacklist = {
        "Heads up!This page uses features your browser doesn’t support. Try a modern browser likeFirefoxorChromefor the best experience.",
        "Copyright ©1999-2025 37signals LLC. All rights reserved.",
        "Back to Basecamp.com",
        "Buy the print edition",
        "Stop Running in Circles and Ship Work that Matters",
        "by Ryan Singer",
        "Chapter 1", "Chapter 2", "Chapter 3", "Chapter 4",
        "Chapter 5", "Chapter 6", "Chapter 7", "Chapter 8",
        "Chapter 9", "Chapter 10", "Chapter 11", "Chapter 12",
        "Chapter 13", "Chapter 14", "Chapter 15",
        "Preface", "Appendices", "Part 1: Shaping", "Part 2: Betting","Part 3: Building"

    }


    # Scraping de cada capítulo
    # chapter_links = soup.select("a[href^='/shapeup/']")  # enlaces de capítulos

    chapter_links = []
    for a in soup.select("a[href^='/shapeup/']"):
        href = a["href"]

        # Omitir índice principal y anchors (#)
        if href == "/shapeup" or "#" in href:
            continue

        # Evitar duplicados
        if href not in chapter_links:
            chapter_links.append(a)

    # return chapter_links
    visited = set()
    for a in chapter_links:
        href = a["href"]
        if href in visited or href == "/shapeup":
            continue
        visited.add(href)

        url = f"https://basecamp.com{href}"
        res = requests.get(url)
        res.encoding = "utf-8"
        chap_soup = BeautifulSoup(res.text, "html.parser")

        # Título del capítulo
        h1 = chap_soup.find("h1")
        if h1:
            story.append(Paragraph(h1.get_text(strip=True), chapter_style))
            story.append(Spacer(1, 12))

        # Subtítulos y párrafos
        for elem in chap_soup.find_all(["h2", "p"]):
            text = elem.get_text(strip=True)

            # Saltar si está en blacklist
            if text in blacklist:
                continue  

            if elem.name == "h2":
                story.append(Paragraph(elem.get_text(strip=True), section_style))
            elif elem.name == "p":
                story.append(Paragraph(elem.get_text(strip=True), content_style))

        story.append(PageBreak())

    # Construir PDF
    doc.build(story)
    print("✅ Libro Shape Up exportado a Shapeup_Basecamp.pdf")


def simplificar_toc(toc: dict) -> dict:
    toc_simplificado = {}
    for seccion, contenidos in toc.items():
        nuevos_contenidos = []
        for item in contenidos:
            if isinstance(item, str):  
                # Nos quedamos solo con strings que no son listas
                nuevos_contenidos.append(item)
            elif isinstance(item, list):  
                # Si es una lista, el "capítulo" ya lo hemos agregado antes (ej: "Chapter 1"),
                # así que la ignoramos
                continue
        toc_simplificado[seccion] = nuevos_contenidos
    return toc_simplificado


new_toc= simplificar_toc(toc)

# Ejecutar función
prueba= scrape_shapeup_to_pdf(new_toc)


✅ Libro Shape Up exportado a Shapeup_Basecamp.pdf
